# Phase 7 & 8: Fine-Tune DistilBERT for Overall Sentiment
## Transformer Sequence Classifier

In this notebook, we fine-tune a pre-trained Transformer model (`distilbert-base-uncased`) for 3-class review sentiment classification.

Workflow:
1. Load processed review dataset
2. Encode labels (`Negative` -> 0, `Neutral` -> 1, `Positive` -> 2)
3. Tokenize text using `DistilBertTokenizer`
4. Fine-tune sequence classifier
5. Evaluate metrics (Accuracy, F1-Score) and compare with baseline
6. Export fine-tuned model and tokenizer to `app/models/trained_model/sentiment/`

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using PyTorch device: {device}")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
data_path = "../data/processed/ecommerce_reviews.csv"
if not os.path.exists(data_path):
    data_path = "data/processed/ecommerce_reviews.csv"

df = pd.read_csv(data_path)
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
id2label = {v: k for k, v in label_map.items()}
df['label'] = df['sentiment'].map(label_map)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(batch):
    return tokenizer(batch['review'], padding='max_length', truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df[['review', 'label']])
test_dataset = Dataset.from_pandas(test_df[['review', 'label']])

train_encoded = train_dataset.map(tokenize_func, batched=True)
test_encoded = test_dataset.map(tokenize_func, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label_map
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./distilbert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=20,
    use_cpu=True  # Force fast CPU execution if no GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
save_dir = "../app/models/trained_model/sentiment"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
with open(os.path.join(save_dir, "label_map.json"), "w") as f:
    json.dump(label_map, f)

print(f"DistilBERT fine-tuned model saved to {save_dir}")